# Aproximações para a duração média de passeios aleatórios

Eduardo Yukio Garrafa Ishihara  
Gustavo Silva Garone  
Elisabeti Kira  
11 de dezembro de 2025

Inspirados na física, propusemos aproximações para a duração de passeios aleatórios diversos, em especial, os de difícil derivação analítica. Para avaliar as aproximações, utilizamos simulações de Monte Carlo na linguagem Julia. Analisamos o desempenho desta linguagem comparando-a com a linguagem Python. Com modelos (passeios) mais sofisticados, utilizamos recursos computacionais para solução de sistemas lineares na construção de novas aproximações, descrevendo o uso de paralelismo e de outras ferramentas. Exibimos e analisamos os resultados obtidos, assim como os algoritmos utilizados.

In [1]:
# Pacotes utilizados no artigo e suas versões
using Pkg
Pkg.add(["Plots", "Distributions", "Statistics", "Random", "LaTeXStrings"])
versioninfo()
Pkg.status()

# Introdução

Neste artigo, estudaremos a duração de passeios aleatórios e suas aproximações. Temos como objetivo descrever modelos de passeios como a “Ruína do Jogador” de Pascal, discutido em @edwards_pascals_1983, generalizando-o para múltiplas dimensões. Abordaremos técnicas analíticas, computacionais e de estimativas para a duração esperada destes passeios.

Começaremos por definir o modelo que será utilizado para descrever os passeios aleatórios, acompanhado de exemplo. Então, apresentaremos o estimador descrito por @ishihara_um_2025, e partiremos para sua avaliação. Primeiramente, compará-lo-emos com resultados probabilísticos estabelecidos na área. Quando avançarmos para passeios em que resultados analíticos não são factíveis, iniciaremos a comparação do estimador com simulações de Monte Carlo.

Descrevemos e justificamos pela literatura a adoção da linguagem de programação *Julia*, essencial para possibilitar a execução eficiente de simulações computacionalmente intensivas em passeios mais complexos.

Em dimensões maiores, forneceremos, até onde é computacionalmente possível, resultados exatos para a duração esperada de passeios aleatórios. Para obter esses resultados, utilizaremos métodos para solução de sistemas lineares extensos simbólicos e empregaremos técnicas de computação paralela, que serão brevemente discutidas.

Este artigo priorizará a descrição do aspecto computacional dessa pesquisa. Leitores são encorajados a consultar a referência @ishihara_um_2025 para uma descrição mais detalhada da construção matemática do estimador e do modelo utilizado.

# Modelando passeios aleatórios

## Modelo

Considere que o jogador $A$ começa com uma fortuna inicial com um total $\$N$ em jogo, $\$a$, $0 \leq a \leq N$, fornecendo um espaço de estados $\{0, 1, \dots, a, \dots, N\}$. Cada rodada produz um resultado aleatório independente $X_i$, $i=1,2,\dots$, representando o ganho ou perda naquela etapa. Isso será chamado de variável “regra”. Após $n$ rodadas, a fortuna do jogador $A$ é dada por

$$S_n = a + \sum_{i=1}^n X_i.$$

A **duração do jogo** é o primeiro tempo $n$ tal que $S_n \leq 0$ (ruína) ou $S_n \geq N$ (vitória). É denotada por $\tau_{X, a, N}$, a regra de parada.

Ao indexarmos $X$ na posição atual do passeio, $X_{n} := X_{S_n}$, podemos estender esse modelo para passeios em meio não homogêneo: definimos $\pmb{R} =
\{X_1, \dots, X_N\}$ e $\tau_{\pmb{R}, a, N}$.

<span class="proof-title">*Comentário*. </span>Neste artigo, não trataremos de passeios em meio não homogêneo no tempo, mas podemos, se preciso, estender esse modelo para incluir essa característica.

Para ajudar no entendimento deste modelo, oferecemos um exemplo:

Considere um jogador apostando em um casino no lançamento de uma moeda honesta. Se a moeda cair em cara, o jogador ganha um dólar; se cair em coroa, perde um dólar. O jogo termina quando o jogador atingir cem dólares ou vá à ruína (seu saldo chegue em zero). O jogador começa com cinquenta dólares.

Podemos modelar esse jogo no modelo da @def-modelodiscreto:

Seja $a = 50$ e $N = 100$. A variável regra do jogo é dada por

$$X = \begin{cases}
-1, & \mathbb{P}(X = -1) = 0.4, \\
+1, & \mathbb{P}(X = +1) = 0.6.
\end{cases}$$

Nota-se que a regra é a mesma para todo estado $S_n$ enquanto o jogo durar, ou seja, o passeio está homogêneo no espaço e tempo.

Após $5$ jogadas, o jogo tomou o seguinte percurso:
$$\pmb{x} = \{x_1, x_2, x_3, x_4, x_5\} = \{1, -1, 1, 1, -1\}.$$

temos então que a posição (fortuna) atual do jogador é
$$S_5 = 50 + (1 - 1 + 1 + 1 -1) = 51.$$

## Jogos unitários e não unitários

Definimos um jogo como **unitário** se a variável regra $X$ assume apenas valores inteiros unitários, ou seja, $X: \Omega \rightarrow \{-1, +1\}$. Caso contrário, o jogo é dito **não unitário**. No @exm-modelodiscreto, o jogo é unitário.

Essa distinção faz-se importante, uma vez que, como apresentaremos na próxima seção, a abordagem analítica para determinar a duração esperada do jogo difere entre jogos unitários e não unitários. De fato, em jogos não unitários, a determinação analítica de $\mathbb{E}(\tau_{X, a, N})$ pode ser muito mais complexa, ou até mesmo inviável.

## Abordagem analítica para determinar $\mathbb{E}(\tau_{X, a, N})$

Estamos interessados na duração esperada do jogo, ou seja, queremos encontrar $\mathbb{E}(\tau_{X, a, N})$. Para jogos unitários, podemos calcular essa esperança por simples condicionamento, como descrito em @stern_conditional_1975.

Com o uso de equações de diferenças finitas, podemos obter esses mesmos resultados de forma mais elegante, como feito por @andel_variance_2012. Desses métodos, obtemos a seguinte expressão para a duração esperada desses jogos:

$$\mathbb{E}(\tau_{X, a, N}) = \begin{cases}
a(N - a), & \mathbb{E}(X) = 0, \\
\frac{a}{q-p}-\frac{N}{q-p} \left( \frac{1- \left( \frac{q}{p} \right)^a}
        {1- \left( \frac{q}{p} \right)^N}\right), & \mathbb{E}(X) \neq 0.
\end{cases}.$$
{#eq-tempoteorico}

em que $p = \mathbb{P}(X = +1)$ e $q = \mathbb{P}(X = -1) = 1 - p$.

Com esse mesmo método, é possível determinar o tempo esperado para alguns outros jogos. Contudo, como descrito por @ishihara_um_2025, a complexidade de resolução do sistema de EDFs cresce exponencialmente. Essa limitação motivou a criação de um estimador para $\mathbb{E}(\tau_{X, a, N})$.

# Aproximando durações de passeios aleatórios

Uma pergunta natural quando se discute aproximações é quanto a sua utilidade. Se possível, é mais desejável utilizar resultados obtidos analiticamente, como, no exemplo da “Ruína do Jogador” clássica, os apresentados na @sec-anali.

Nos cenários em que não é prático o desenvolvimento analítico, soluções computacionais são buscadas. Estas soluções podem fornecer resultados teóricos corretos, como na solução de sistemas lineares extensos que abordaremos na \[SESSÃO AQUI\], ou aproximações pelo método de Monte Carlo, como descreve @harrison_introduction_2010.

Ainda assim, a simulação também traz desvantagens para além da perda de precisão e de interpretabilidade. Por exemplo, como abordado em @ritter_determining_2011, um número considerável de simulações é necessário para garantir acurácia dos dados e estabilidade. Diante disso, vê-se justificável a busca de estimadores eficientes para a duração esperada de passeios aleatórios diversos.

## Um estimador heurístico

Baseando-se na mecânica clássica, @ishihara_um_2025 propôs um estimador para passeios não unitários em meio homogêneo:

Seja um jogo definido conforme a @def-modelodiscreto, com variável regra $X$ tal que $\mathbb{E}(X) \neq 0$, valores $a$ e $N$. O estimador proposto para a duração esperada desse jogo é dado por:

$$\hat{\mathbb{E}}(\tau_{X, a N}) = \begin{cases}
\frac{N-a}{\mathbb{E}(X)},  & \mathbb{E}(X) > 0, \\
\left \lvert \frac{a}{\mathbb{E}(X)} \right \rvert,  & \mathbb{E}(X) < 0.
\end{cases}$$

Primeiramente, o estimador da @def-estimadorsimples foi comparado com o resultado da @eq-tempoteorico para jogos unitários. Retomando o @exm-modelodiscreto, temos, como $\mathbb{E}(X) = 0.2$:

$$\begin{aligned}
\hat{\mathbb{E}}(\tau_{X, 50, 100}) = \frac{100}{0.2} = 250 \\
\mathbb{E}(\tau_{X, 50, 100}) = \frac{50}{0.6 - 0.4} - \frac{100}{0.6 - 0.4}
\left( \frac{1 - \left( \frac{0.4}{0.6} \right)^{50}}{1 - \left( \frac{0.4}{0.6}
\right)^{100}} \right) = 250
\end{aligned}$$

Partimos para testá-lo em outros casos unitários. Dos nossos resultados, obtemos, fixada a fortuna inicial $a=50$ e total $N$, e variando a probabilidade de ganho $p$:

In [1]:
using Plots, LaTeXStrings

function estimador_simples(E_X, a, N)
    if E_X > 0
        return (N - a) / E_X
    elseif E_X < 0
        return - a / E_X
    else
        return NaN
    end
end

function tempo_teorico_unitario(a, N, p)
    q = 1 - p
    if p == q
        return a * (N - a)
    else
        return (a / (q - p)) - (N / (q - p)) * ((1 - (q / p)^a) / (1 - (q / p)^N))
    end
end

function comparar_estimador_unitario(a, N)
    ps = 0:0.01:1
    m = length(ps)
    estimados = Float64[]
    teoricos = Float64[]
    for p in ps
        E_X = (1 * p) + (-1 * (1 - p))
        push!(estimados, estimador_simples(E_X, a, N))
        push!(teoricos, tempo_teorico_unitario(a, N, p))
    end
    plt = plot(
        ps, estimados, label = "Duração estimada",
        xlabel = "Probabilidade de vitória (p)", ylabel = "Duração",
        legend = :topright,
    )
    plot!(ps, teoricos, label = "Duração esperada", linestyle = :dash)
    vline!(
        [0.48], label = L"p = 0.48 \Rightarrow \mathbb{E}(X) = -0.04",
        linestyle = :dashdot
    )
    vline!(
        [0.52], label = L"p = 0.52 \Rightarrow \mathbb{E}(X) = +0.04",
        linestyle = :dot
    )
    return plt
end

display(comparar_estimador_unitario(50, 100))

Notamos que o estimador apresenta bom corpotamento nessa classe de jogos. Uma inadequação de seu uso existe para casos em que $\mathbb{E}(X)$ se aproxima de $0$, como indica o gráfico da @fig-comparacaoestimadorxteorico no intervalo $[0.48, 0.52]$

Validado o estimador para grande parte dos casos mais simples, partimos para análise em jogos não unitários. Para isso, utilizamos simulações de Monte Carlo para obter valores aproximados de $\mathbb{E}(\tau_{X, a, N})$. Apresentamos no @algo-montecarlosimples uma implementação simples desse processo @harrison_introduction_2010, @ritter_determining_2011.

``` pseudocode
#| label: algo-montecarlosimples
#| html-indent-size: "1.2em"
#| html-comment-delimiter: "//"
#| html-line-number: true
#| html-line-number-punc: ":"
#| html-no-end: false
#| pdf-placement: "htb!"
#| pdf-line-number: true

\begin{algorithm}
\caption{Monte Carlo Simples}
\begin{algorithmic}
\Procedure{MonteCarlo}{$MC, p, g, l, a, N$}
    \State $total\_tempo \gets 0$
    \For{$i =1$ to $MC$}
        \State $pos$ \gets $a$
        \While{$pos > 0$ and $pos < N$}
            \State $u \gets \Call{Sortear}{U(0,1)}$
            \If{$u < p$}
                \State $pos$ \gets $pos + g$
            \Else
                \State $pos$ \gets $pos - l$
            \EndIf
            \State $total\_tempo \gets total\_tempo + 1$
        \EndWhile
    \EndFor
    \State \Return $total\_tempo / MC$
\EndProcedure
\end{algorithmic}
\end{algorithm}
```

In [1]:
function monte_carlo_simples(MC, p, g, l, a, N)
    total_tempo = 0
    for i in 1:MC
        pos = a
        while pos > 0 && pos < N
            u = rand()
            if u < p
                pos += g
            else
                pos -= l
            end
            total_tempo += 1
        end
    end
    return total_tempo / MC
end

function comparar_estimador_montecarlo(MC, p, g, l, a, N)
    ps = 0:0.01:1
    m = length(ps)
    estimados = Float64[]
    simulados = Float64[]
    for p in ps
        E_X = (g * p) + (-l * (1 - p))
        push!(estimados, estimador_simples(E_X, a, N))
        push!(simulados, monte_carlo_simples(MC, p, g, l, a, N))
    end
    plt = plot(
        ps, estimados, label = "Duração estimada",
        xlabel = "Probabilidade de vitória (p)", ylabel = "Duração",
        legend = :topright, ylims = (0, 2500)
    )
    plot!(ps, simulados, label = "Duração simulada", linestyle = :dash)
    return plt
end

display(comparar_estimador_montecarlo(10_000, 0.5, 2, 1, 50, 100))

Para jogos não unitários, o estimador também apresenta bom desempenho, como mostrado na @fig-comparacaoestimadorxsimulado. Novamente, observa-se uma queda na precisão do estimador à medida que $\mathbb{E}(X)$ se aproxima de $0$.

Com esses resultados, validamos o estimador proposto na @def-estimadorsimples para passeios aleatórios em meio homogêneo, tanto unitários quanto não unitários. Dessa forma, partimos para a análise de passeios mais complexos.

## Meio não-homogêneo

Ao tratarmos de passeios em meio não-homogêneo, precisamos descrever o comportamento deste ambiente. No geral, podemos modelar um meio aleatório ou deterinístico. Em ambos casos, a variável regra $X$ dependerá da posição atual do passeio, ou seja, $X = X_{S_n}$. Entretanto, enquanto no meio aleatório pode ser modelado



# Referências